# Bài 30: Phép chiếu &mdash; Ý tưởng trung tâm (Projection &mdash; The Central Idea)

Đây là phần mở đầu của Phần 3 của khóa học: học sâu cho thị giác máy tính. Trước khi viết bất kỳ mạng nơ-ron nào, đáng để dừng lại suy ngẫm về một ý tưởng hiếm khi được chú ý nhiều dù có tầm quan trọng nền tảng. **Phép chiếu (projection)** là một ý tưởng đơn giản: ánh xạ các điểm từ một không gian sang một không gian khác, thường là đơn giản hơn, bằng một tổ hợp tuyến tính các tọa độ của chúng. Phép chiếu lặng lẽ chạy xuyên suốt phần lớn Phần 1 và 2 &mdash; nhiều kỹ thuật từ đầu đến giờ thực ra đều là phép chiếu. Phép chiếu cũng là nền tảng của mọi thứ từ đây trở đi &mdash; mạng nơ-ron hóa ra chẳng qua là các phép chiếu *được học* và *được ghép nối*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Hai phép chiếu bạn đã xây dựng

**PCA (Bài 6).** Cho một đám mây điểm, các vector riêng của ma trận hiệp phương sai của chúng cho ra các hướng để chiếu lên. Chiếu lên vector riêng hàng đầu, $y = v^\top x$, thu gọn mỗi điểm 2D thành một con số duy nhất &mdash; tọa độ dọc theo hướng có độ trải lớn nhất.

In [ ]:
rng = np.random.default_rng(1)
cloud = rng.multivariate_normal([0, 0], [[3, 1.5], [1.5, 1]], 100)

cov = np.cov(cloud.T)
eigvals, eigvecs = np.linalg.eigh(cov)
principal_direction = eigvecs[:, -1]  # eigenvector of the largest eigenvalue

projected = cloud @ principal_direction

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].scatter(cloud[:, 0], cloud[:, 1], s=15, alpha=0.6)
axes[0].plot([0, 3 * principal_direction[0]], [0, 3 * principal_direction[1]], color='red', linewidth=2)
axes[0].set_aspect('equal')
axes[0].set_title('2D cloud + principal direction')
axes[1].scatter(projected, np.zeros_like(projected), s=15, alpha=0.6)
axes[1].set_yticks([])
axes[1].set_title('Projected onto that direction (1D)')
plt.tight_layout()
plt.show()

**Phép chiếu camera (Bài 24, 25).** $P = K[R|t]$ ánh xạ một điểm 3D trong thế giới thành một điểm ảnh 2D. Mỗi hàng của ma trận đó, trước phép chia thuần nhất cuối cùng, tự nó là một phép chiếu tuyến tính: một tích vô hướng giữa tọa độ của điểm và một hướng cố định, cộng với một độ lệch.

Cả hai ví dụ có chung cơ chế: chọn một hướng $w$ (và có thể một độ lệch $b$), rồi tính $y = w^\top x + b$. Điều khác nhau là *tại sao* hướng đó được chọn: PCA chọn $w$ để cực đại hóa độ trải của dữ liệu đã chiếu (một mục tiêu không giám sát, thuần túy hình học). Các hàng của camera được cố định bởi hình học vật lý của nó. Không cái nào được chọn để giải một bài toán phân loại hay nhận dạng &mdash; nhưng không có gì ngăn ta chọn $w$ cho đúng mục đích đó.

## Phép nhân ma trận: nhiều phép chiếu cùng lúc

Mọi thứ ở trên đều tính một con số $y = w^\top x$ từ một hướng $w$. Xếp $m$ hướng khác nhau thành các hàng của một ma trận $W$ (kích thước $m \times n$), thì $y = Wx$ tính cả $m$ phép chiếu cùng lúc &mdash; mỗi hàng một tích vô hướng. Đây chính là bản chất của phép nhân ma trận: một dàn các phép chiếu đồng thời. Điều đó cũng có nghĩa là một số phép toán tuyến tính trông có vẻ khác nhau ở phần trước của khóa học thực ra là cùng một phép toán, chỉ với một lựa chọn $W$ khác nhau, có cấu trúc:

- **Tích chập** (Bài 10): mỗi giá trị đầu ra là một tích vô hướng giữa một vùng lân cận nhỏ và một kernel &mdash; một phép chiếu. Trượt kernel qua toàn bộ tín hiệu là một phép nhân với một ma trận thưa có các hàng là các bản sao dịch chuyển của kernel (một ma trận Toeplitz).
- **Biến đổi Fourier** (Bài 15): mỗi giá trị đầu ra là một tích vô hướng giữa *toàn bộ* tín hiệu và một hình sin ở một tần số cố định &mdash; một phép chiếu lên tần số đó. Phép biến đổi đầy đủ là một phép nhân với một ma trận cố định gồm các sin và cosin.
- **Một phép đồng nhất hoặc ma trận chiếu camera** (Bài 24, 25): mỗi tọa độ đầu ra là một tích vô hướng giữa một điểm thuần nhất và một hàng của $H$ hoặc $P$.

Ta kiểm chứng trực tiếp hai cái đầu tiên: dựng ma trận bằng tay, và kiểm tra nó tái tạo chính xác `np.convolve`/`np.fft.fft`.

In [ ]:
signal = rng.normal(size=10)
kernel = np.array([0.25, 0.5, 0.25])  # a small smoothing kernel, as in Lesson 10

# a matrix whose rows are shifted copies of the kernel -- a Toeplitz matrix
n, k = len(signal), len(kernel)
W_conv = np.zeros((n - k + 1, n))
for i in range(n - k + 1):
    W_conv[i, i:i + k] = kernel

via_matmul = W_conv @ signal
via_convolve = np.convolve(signal, kernel, mode='valid')
print(f'max diff, convolution as a matrix multiply: {np.abs(via_matmul - via_convolve).max():.2e}')

N = 8
idx = np.arange(N)
W_dft = np.exp(-2j * np.pi * np.outer(idx, idx) / N)  # row k: the frequency-k sinusoid, sampled at every point

x = rng.normal(size=N)
via_matmul = W_dft @ x
via_fft = np.fft.fft(x)
print(f'max diff, DFT as a matrix multiply:          {np.abs(via_matmul - via_fft).max():.2e}')

Một lớp mạng nơ-ron kết nối đầy đủ (fully-connected) chẳng qua chính là $y = Wx + b$ này, với các hàng của $W$ được *học* để giải một nhiệm vụ thay vì được cố định bởi cấu trúc suy ra thủ công (một hình dạng kernel, một dàn hình sin, hình học của một camera).

Ảnh động này tính $y=W_\text{conv}x$ từng hàng một: mỗi bước làm nổi bật một hàng của $W$, cửa sổ tương ứng mà nó mở lên $x$, và tích vô hướng duy nhất mà hàng đó tính, điền vào một phần tử của $y$ trước khi chuyển sang hàng tiếp theo. Từng hàng một, đây chính xác là cửa sổ trượt của một phép tích chập &mdash; và tương đương với nhiều phép chiếu chạy nối tiếp nhau.

<img src="../animations/matrix_as_projections_demo.gif" alt="Ảnh động tính tích ma trận-vector từng hàng, làm nổi bật mỗi lần một hàng của ma trận Toeplitz tích chập, cửa sổ tín hiệu mà nó lấy tích vô hướng, và phần tử đầu ra tương ứng được điền vào" style="display: block; margin: 0 auto; max-width: 680px;">

Để trực quan hóa một phép chiếu đơn lẻ (Hàng 0), lưu ý rằng kernel $w=[0.25,0.5,0.25]$ xác định một hướng trong 3D, và cửa sổ 3 phần tử $(x_0,x_1,x_2)$ xác định một điểm trong cùng không gian 3D đó. Chiếu điểm đó lên tia &mdash; hạ đường vuông góc &mdash; rơi vào một khoảng cách có dấu $s$ dọc theo tia; co giãn $s$ theo độ dài của chính $w$ trả lại đúng $y_0=w\cdot x = w^\top x = |w|s$, cùng con số mà hàng 0 ở trên tạo ra.

<img src="../animations/projection_as_3d_ray_demo.gif" alt="Ảnh động từng bước cho thấy một kernel 3 phần tử là một hướng trong 3D, một cửa sổ tín hiệu là một điểm trong không gian đó, hình chiếu vuông góc của nó lên tia, và khoảng cách có dấu thu được bằng tích vô hướng y0" style="display: block; margin: 0 auto; max-width: 640px;">

## Một phép chiếu cho bài toán phân loại

Giả sử thay vì "cực đại hóa độ trải", mục tiêu là "tách hai lớp". Cùng công thức $y = w^\top x + b$ vẫn áp dụng &mdash; chiếu mỗi điểm lên một hướng $w$, và phân loại theo *dấu* của điểm số thu được. Đây, xét toàn bộ, chính là một nơ-ron nhân tạo đơn lẻ không có hàm kích hoạt: phần lõi tuyến tính mà mọi lớp mạng nơ-ron đều được xây dựng từ đó.

In [ ]:
class_a = rng.normal(loc=[-2, -1], scale=0.8, size=(60, 2))
class_b = rng.normal(loc=[2, 1.5], scale=0.8, size=(60, 2))

# a principled hand-picked direction: point from one class's mean toward the other's
w = class_b.mean(axis=0) - class_a.mean(axis=0)
w = w / np.linalg.norm(w)
midpoint = (class_a.mean(axis=0) + class_b.mean(axis=0)) / 2
b = -w @ midpoint  # threshold: the boundary passes through the midpoint between the classes

score_a = class_a @ w + b
score_b = class_b @ w + b
accuracy = (np.sum(score_a < 0) + np.sum(score_b > 0)) / (len(score_a) + len(score_b))
print(f'w = {np.round(w, 3)}, b = {b:.3f}')
print(f'classification accuracy: {accuracy:.1%}')

In [ ]:
def plot_projection_classifier(ax, class_a, class_b, w, b, title):
    ax.scatter(*class_a.T, s=15, label='class A')
    ax.scatter(*class_b.T, s=15, label='class B')

    # decision boundary: the line {x : w.x + b = 0}, drawn through its closest point to the origin
    foot = -b * w
    perp = np.array([-w[1], w[0]])
    p1, p2 = foot + 5 * perp, foot - 5 * perp
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], color='black', linewidth=1.5, label='decision boundary')
    ax.arrow(*foot, *w, head_width=0.15, color='red', length_includes_head=True, label='w')

    ax.set_aspect('equal')
    ax.set_title(title, fontsize=9)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_projection_classifier(axes[0], class_a, class_b, w, b, f'Decision boundary\naccuracy={accuracy:.0%}')
axes[0].legend(fontsize=7)

axes[1].scatter(score_a, np.zeros_like(score_a), s=15, label='class A')
axes[1].scatter(score_b, np.zeros_like(score_b), s=15, label='class B')
axes[1].axvline(0, color='black', linewidth=1.5, label='threshold')
axes[1].set_yticks([])
axes[1].set_title('The same points, projected to 1D')
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

Hai góc nhìn về chính xác cùng một phép toán: bên trái, $w$ là một hướng trong không gian 2D ban đầu và ranh giới quyết định (decision boundary) là đường thẳng vuông góc với nó; bên phải, mọi điểm đã được chiếu xuống hướng đó, và việc phân loại chỉ là phân ngưỡng một con số duy nhất tại 0. Hình ảnh 2D trực quan hơn, nhưng hình ảnh 1D mới là thứ thực sự tổng quát hóa được &mdash; trong 100 chiều không có hình nào để vẽ "ranh giới", nhưng "chiếu thành một con số duy nhất, rồi phân ngưỡng" vẫn hoạt động y hệt.

## Điều mà một phép chiếu đơn lẻ không làm được

Một phép chiếu tuyến tính đơn lẻ chỉ có thể tạo ra ranh giới quyết định là *đường thẳng* (một siêu phẳng (hyperplane), trong không gian nhiều chiều hơn). Một số tập dữ liệu hoàn toàn không có đường thẳng phân tách nào, bất kể chọn $w$ và $b$ thế nào &mdash; ví dụ kinh điển là một lớp bao quanh lớp kia.

In [ ]:
theta_in = rng.uniform(0, 2 * np.pi, 60)
inner = np.stack([0.5 * np.cos(theta_in), 0.5 * np.sin(theta_in)], axis=1) + rng.normal(0, 0.1, (60, 2))
theta_out = rng.uniform(0, 2 * np.pi, 60)
outer = np.stack([2.0 * np.cos(theta_out), 2.0 * np.sin(theta_out)], axis=1) + rng.normal(0, 0.15, (60, 2))

# search many directions and thresholds for the best possible LINEAR separator
best_acc, best_w, best_b = 0, None, None
for _ in range(2000):
    w_try = rng.normal(size=2)
    w_try /= np.linalg.norm(w_try)
    for b_try in np.linspace(-3, 3, 61):
        s_in, s_out = inner @ w_try + b_try, outer @ w_try + b_try
        acc = max((s_in < 0).sum() + (s_out > 0).sum(), (s_in > 0).sum() + (s_out < 0).sum()) / 120
        if acc > best_acc:
            best_acc, best_w, best_b = acc, w_try, b_try

print(f'best achievable accuracy with ANY single linear projection: {best_acc:.1%}')

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_projection_classifier(ax, inner, outer, best_w, best_b, f'Best possible linear boundary\naccuracy={best_acc:.0%}')
plt.show()

Không có phép xoay hay dịch chuyển nào của một đường thẳng đơn lẻ có thể tách một vành khuyên khỏi hình đĩa mà nó bao quanh &mdash; điều tốt nhất mà bất kỳ phép chiếu tuyến tính nào làm được là một sự thỏa hiệp tầm thường. Đây là bức tường mà mọi phương pháp thuần tuyến tính đều đâm vào, và đó chính là động lực cho hai bài tiếp theo: trước tiên, *học* $w$ và $b$ một cách tự động thay vì chọn thủ công (Bài 31), và sau đó, quan trọng hơn, *ghép nối nhiều phép chiếu với các hàm phi tuyến xen giữa* (Bài 32), thứ mà với đủ năng lực, có thể tách các vùng có hình dạng về cơ bản là bất kỳ.

Ảnh động này quét $w$ qua mọi hướng (với ngưỡng $b$ tốt nhất có thể ở mỗi góc) trên cùng dữ liệu vành-khuyên-và-đĩa. Hãy quan sát ranh giới quyết định di chuyển quanh trong khi không có phép xoay hay dịch chuyển nào tiến gần tới việc tách được các lớp.

<img src="../animations/single_projection_sweep_demo.gif" alt="Ảnh động quét một ranh giới quyết định tuyến tính đơn lẻ qua mọi góc trên tập dữ liệu vành-khuyên-và-đĩa, với độ chính xác vẫn tầm thường bất kể góc hay ngưỡng" style="display: block; margin: 0 auto; max-width: 720px;">

### Bài tập

1. Trong phần ôn tập PCA, chiếu `cloud` lên vector riêng *thứ hai* (nhỏ hơn) thay vì vector chính. Độ trải của hình chiếu 1D thu được so với khi chiếu lên hướng chính thế nào, và tại sao điều đó khớp với những gì bản thân giá trị riêng cho bạn biết (Bài 6)?
2. Sửa trung bình và độ trải của `class_a`/`class_b` để các lớp chồng lấn nhiều hơn. Đến mức nào thì hướng hiệu-trung-bình `w` không còn đạt độ chính xác cao, và bạn có thể tự tìm ra một `w` *tốt hơn* hướng hiệu-trung-bình cho trường hợp khó hơn đó không?
3. Với tập dữ liệu vành-khuyên-và-đĩa, thay vì phép chiếu tuyến tính, hãy thử phân loại bằng một hàm phi tuyến đơn giản của các điểm: khoảng cách tới gốc tọa độ, $r = \sqrt{x_1^2+x_2^2}$, phân ngưỡng tại một giá trị nào đó. Đặc trưng phi tuyến duy nhất này đạt độ chính xác bao nhiêu, và điều đó gợi ý gì về *loại* phép chiếu thực sự giải được bài toán này?